In [1]:
!python --version
!pip -q install --upgrade google-cloud-aiplatform google-cloud-storage tensorflow==2.11.* scikit-learn pillow tqdm

Python 3.10.18


In [2]:
import os, json, numpy as np, pandas as pd, tensorflow as tf
from google.cloud import aiplatform, storage
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm

2025-11-12 11:40:05.903818: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-12 11:40:08.903120: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2025-11-12 11:40:08.903359: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local

In [3]:
PROJECT_ID = "com3021-fmnist"
REGION     = "europe-west2"
ENDPOINT_ID= "4968591890849988608"

GCS_BUCKET = "com3021-fmnist-rps210-bucket"
DATA_DIR   = "/home/jupyter/fmnist-dataset"
MODEL_DIR  = "/home/jupyter/fashion-mnist2"

In [4]:
IMG_SIZE = (28, 28)

def load_label_csv(csv_path):
    df = pd.read_csv(csv_path)
    if 'label' not in df.columns:
        raise ValueError("CSV must include a 'label' column.")
    return df

def load_image_as_array(path):
    img = Image.open(path).convert("L").resize(IMG_SIZE)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr

def build_split(split_dir, csv_path):
    df = load_label_csv(csv_path)
    xs, ys = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        xs.append(load_image_as_array(os.path.join(split_dir, row['filename'])))
        ys.append(int(row['label']))
    X = np.stack(xs, axis=0)
    y = np.array(ys, dtype=np.int64)
    return X, y

TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR  = os.path.join(DATA_DIR, "test")

X_test, y_test = build_split(TEST_DIR, os.path.join(DATA_DIR, "test.csv"))
X_train, y_train = build_split(TRAIN_DIR, os.path.join(DATA_DIR, "train.csv"))
len(X_train), len(X_test), X_train.shape, y_train.shape

100%|██████████| 60000/60000 [00:49<00:00, 1202.00it/s]


(60000, 10000, (60000, 28, 28), (60000,))

In [5]:
aiplatform.init(project=PROJECT_ID, location=REGION)
endpoint = aiplatform.Endpoint(
    endpoint_name=f"projects/{PROJECT_ID}/locations/{REGION}/endpoints/{ENDPOINT_ID}"
)


def predict_batch_endpoint(X, start_bs=64, byte_cap=1_300_000):
    bs = min(start_bs, len(X))
    while bs > 1:
        probe = [np.round(x, 4).tolist() for x in X[:bs]]   # rounding shrinks JSON
        approx_bytes = len(json.dumps({"instances": probe}))
        if approx_bytes <= byte_cap:
            break
        bs //= 2
    if bs == 1:
        pass

    preds = []
    for i in range(0, len(X), bs):
        chunk = X[i:i+bs]
        instances = [np.round(x, 4).tolist() for x in chunk]
        resp = endpoint.predict(instances=instances)
        probs = np.array(resp.predictions, dtype=np.float32)
        preds.extend(probs.argmax(axis=1))
    return np.array(preds, dtype=np.int64)

y_pred_deployed = predict_batch_endpoint(X_test, start_bs=32)
acc_deployed = (y_pred_deployed == y_test).mean()
print(f"Deployed endpoint accuracy on test set: {acc_deployed:.4f}")

Deployed endpoint accuracy on test set: 0.8779


In [6]:
local_model = tf.keras.models.load_model(MODEL_DIR)
local_model.summary()

X_test_tf = X_test[..., np.newaxis]

probs_local = local_model.predict(X_test_tf, verbose=0)
y_pred_local = probs_local.argmax(axis=1)

acc_local = accuracy_score(y_test, y_pred_local)
print(f"Local SavedModel accuracy: {acc_local:.4f}")

print("\nClassification report (local):")
print(classification_report(y_test, y_pred_local, digits=4))

cm_local = confusion_matrix(y_test, y_pred_local)
cm_local

2025-11-12 11:42:41.506508: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2025-11-12 11:42:41.506645: W tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: UNKNOWN ERROR (303)
2025-11-12 11:42:41.506725: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (fmnist-workbench-rps210): /proc/driver/nvidia/version does not exist
2025-11-12 11:42:41.507784: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense (Dense)               (None, 128)               100480    
                                                                 
 dense_1 (Dense)             (None, 10)                1290      
                                                                 
Total params: 101,770
Trainable params: 101,770
Non-trainable params: 0
_________________________________________________________________
Local SavedModel accuracy: 0.8803

Classification report (local):
              precision    recall  f1-score   support

           0     0.8069    0.8610    0.8331      1000
           1     0.9800    0.9820    0.9810      1000
           2     0.8130    0.7650    0.7883      1000
           3     0.9034    0.87

array([[861,   2,  14,  21,   8,   3,  82,   0,   9,   0],
       [  3, 982,   1,   7,   5,   0,   1,   0,   1,   0],
       [ 26,   1, 765,   8, 108,   1,  89,   0,   2,   0],
       [ 30,  10,  16, 870,  44,   1,  23,   0,   5,   1],
       [  2,   1,  62,  32, 822,   0,  78,   0,   3,   0],
       [  1,   1,   0,   1,   0, 940,   0,  29,   4,  24],
       [139,   5,  81,  20,  67,   1, 674,   0,  13,   0],
       [  0,   0,   0,   0,   0,   9,   0, 942,   1,  48],
       [  4,   0,   2,   4,   6,   3,   6,   3, 972,   0],
       [  1,   0,   0,   0,   0,   6,   0,  18,   0, 975]])

In [7]:
print(f"Deployed endpoint accuracy: {acc_deployed:.4f}")
print(f"Local SavedModel accuracy:  {acc_local:.4f}")

print("\nClassification report (deployed):")
print(classification_report(y_test, y_pred_deployed, digits=4))

cm_deployed = confusion_matrix(y_test, y_pred_deployed)

cm_local, cm_deployed

Deployed endpoint accuracy: 0.8779
Local SavedModel accuracy:  0.8803

Classification report (deployed):
              precision    recall  f1-score   support

           0     0.8219    0.8350    0.8284      1000
           1     0.9898    0.9680    0.9788      1000
           2     0.8402    0.7100    0.7696      1000
           3     0.9090    0.8590    0.8833      1000
           4     0.7515    0.8560    0.8004      1000
           5     0.9763    0.9470    0.9614      1000
           6     0.6783    0.7020    0.6899      1000
           7     0.9508    0.9470    0.9489      1000
           8     0.9436    0.9870    0.9648      1000
           9     0.9398    0.9680    0.9537      1000

    accuracy                         0.8779     10000
   macro avg     0.8801    0.8779    0.8779     10000
weighted avg     0.8801    0.8779    0.8779     10000



(array([[861,   2,  14,  21,   8,   3,  82,   0,   9,   0],
        [  3, 982,   1,   7,   5,   0,   1,   0,   1,   0],
        [ 26,   1, 765,   8, 108,   1,  89,   0,   2,   0],
        [ 30,  10,  16, 870,  44,   1,  23,   0,   5,   1],
        [  2,   1,  62,  32, 822,   0,  78,   0,   3,   0],
        [  1,   1,   0,   1,   0, 940,   0,  29,   4,  24],
        [139,   5,  81,  20,  67,   1, 674,   0,  13,   0],
        [  0,   0,   0,   0,   0,   9,   0, 942,   1,  48],
        [  4,   0,   2,   4,   6,   3,   6,   3, 972,   0],
        [  1,   0,   0,   0,   0,   6,   0,  18,   0, 975]]),
 array([[835,   2,  13,  11,   3,   2, 119,   0,  15,   0],
        [  3, 968,   1,  20,   2,   0,   3,   0,   3,   0],
        [ 17,   1, 710,   9, 150,   0, 107,   0,   6,   0],
        [ 37,   5,   7, 859,  44,   0,  43,   0,   5,   0],
        [  0,   1,  58,  22, 856,   0,  59,   0,   4,   0],
        [  0,   0,   0,   1,   0, 947,   0,  19,   5,  28],
        [122,   1,  56,  19,  81,   1,

### 3.4 Discussion

The local SavedModel and the deployed endpoint show very similar overall accuracy (approximately 0.88), indicating that the core model behaviour remains consistent across environments. The small difference can be explained by the difference in inference execution. The local model is evaluated directly in TensorFlow within the notebook, whereas the deployed model runs inside a Vertex AI TensorFlow Serving environment, where computation graphs, kernel optimisations, and floating-point execution may differ. Additionally, the endpoint receives input through JSON, and the rounding applied to meet request size limits can introduce slight variation in the probability outputs.

The confusion matrices for both models show the same patterns of misclassification. The most prominent confusions occur between visually similar clothing categories:

- Class 0 (T-shirt/Top) and Class 6 (Shirt)
- Class 2 (Pullover) and Class 4 (Coat)

These items share similar silhouettes when reduced to 28×28 grayscale resolution, which reduces the distinct texture and structural cues the model would otherwise rely on. As a result, the ambiguity is inherent to the dataset rather than a limitation of the deployment process.

Because both models produce nearly identical misclassification patterns, the observed performance difference is attributed to expected differences in inference environment rather than any preprocessing or architectural inconsistency.


In [8]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

def make_ds(X, y, batch=256, shuffle=False, augment=False):
    X = X[..., np.newaxis].astype("float32")  # (N,28,28,1)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(8192, reshuffle_each_iteration=True)
    if augment:
        aug = tf.keras.Sequential([
            tf.keras.layers.RandomTranslation(0.05, 0.05, fill_mode="nearest"),
            tf.keras.layers.RandomRotation(0.05, fill_mode="nearest"),
        ])
        ds = ds.map(lambda xi, yi: (aug(xi, training=True), yi), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(X_tr,  y_tr,  batch=256, shuffle=True,  augment=True)
val_ds   = make_ds(X_val, y_val, batch=256, shuffle=False, augment=False)
test_ds  = make_ds(X_test, y_test, batch=256, shuffle=False, augment=False)

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


In [9]:
def build_cnn(drop=0.25):
    inputs = tf.keras.Input(shape=(28,28,1))
    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Dropout(drop)(x)

    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Dropout(drop)(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(drop)(x)
    outputs = tf.keras.layers.Dense(10, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model_scratch = build_cnn()
model_scratch.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 conv2d (Conv2D)             (None, 28, 28, 32)        320       
                                                                 
 conv2d_1 (Conv2D)           (None, 28, 28, 32)        9248      
                                                                 
 max_pooling2d (MaxPooling2D  (None, 14, 14, 32)       0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 14, 14, 32)        0         
                                                                 
 conv2d_2 (Conv2D)           (None, 14, 14, 64)        18496     
                                                             

In [10]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=3, min_lr=1e-5),
]

history = model_scratch.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    verbose=2,
    callbacks=callbacks
)

Epoch 1/30
211/211 - 238s - loss: 0.7799 - accuracy: 0.7071 - val_loss: 0.4506 - val_accuracy: 0.8303 - lr: 0.0010 - 238s/epoch - 1s/step
Epoch 2/30
211/211 - 240s - loss: 0.5134 - accuracy: 0.8070 - val_loss: 0.3706 - val_accuracy: 0.8567 - lr: 0.0010 - 240s/epoch - 1s/step
Epoch 3/30
211/211 - 243s - loss: 0.4485 - accuracy: 0.8318 - val_loss: 0.3409 - val_accuracy: 0.8695 - lr: 0.0010 - 243s/epoch - 1s/step
Epoch 4/30
211/211 - 226s - loss: 0.4063 - accuracy: 0.8487 - val_loss: 0.3072 - val_accuracy: 0.8877 - lr: 0.0010 - 226s/epoch - 1s/step
Epoch 5/30
211/211 - 228s - loss: 0.3772 - accuracy: 0.8607 - val_loss: 0.2863 - val_accuracy: 0.8952 - lr: 0.0010 - 228s/epoch - 1s/step
Epoch 6/30
211/211 - 231s - loss: 0.3526 - accuracy: 0.8697 - val_loss: 0.2745 - val_accuracy: 0.8982 - lr: 0.0010 - 231s/epoch - 1s/step
Epoch 7/30
211/211 - 233s - loss: 0.3384 - accuracy: 0.8736 - val_loss: 0.2577 - val_accuracy: 0.9063 - lr: 0.0010 - 233s/epoch - 1s/step
Epoch 8/30
211/211 - 240s - loss: 

In [11]:
test_loss, test_acc = model_scratch.evaluate(test_ds, verbose=0)
print(f"Scratch model test accuracy: {test_acc:.4f}")

probs_scratch = np.concatenate([model_scratch.predict(batch[0], verbose=0) for batch in test_ds], axis=0)
y_pred_scratch = probs_scratch.argmax(axis=1)

Scratch model test accuracy: 0.9197


In [12]:
print("\nClassification report (scratch):")
print(classification_report(y_test, y_pred_scratch, digits=4))

cm_scratch = confusion_matrix(y_test, y_pred_scratch)
cm_scratch


Classification report (scratch):
              precision    recall  f1-score   support

           0     0.8839    0.8600    0.8718      1000
           1     0.9870    0.9880    0.9875      1000
           2     0.8711    0.8850    0.8780      1000
           3     0.9022    0.9320    0.9169      1000
           4     0.8796    0.8330    0.8557      1000
           5     0.9821    0.9900    0.9861      1000
           6     0.7561    0.7750    0.7654      1000
           7     0.9710    0.9700    0.9705      1000
           8     0.9871    0.9910    0.9890      1000
           9     0.9789    0.9730    0.9759      1000

    accuracy                         0.9197     10000
   macro avg     0.9199    0.9197    0.9197     10000
weighted avg     0.9199    0.9197    0.9197     10000



array([[860,   0,  12,  18,   4,   0, 101,   0,   5,   0],
       [  0, 988,   0,   9,   1,   0,   1,   0,   1,   0],
       [ 14,   1, 885,   8,  36,   0,  56,   0,   0,   0],
       [  8,   9,  10, 932,  15,   0,  24,   0,   2,   0],
       [  0,   0,  62,  38, 833,   0,  66,   0,   1,   0],
       [  0,   0,   0,   0,   0, 990,   0,   8,   0,   2],
       [ 89,   2,  47,  27,  56,   0, 775,   0,   4,   0],
       [  0,   0,   0,   0,   0,  11,   0, 970,   0,  19],
       [  2,   1,   0,   1,   2,   1,   1,   1, 991,   0],
       [  0,   0,   0,   0,   0,   6,   1,  20,   0, 973]])

### 4.1 Notes

Training converged steadily over 30 epochs with ReduceLROnPlateau lowering the LR twice (to 5e-4 at epoch 21 and 2.5e-4 at epoch 29). Validation accuracy reached \~0.93 and the final test accuracy is 0.9223. Compared with the two provided models, this model performs \~4.4–4.6 percentage points higher (0.922 vs \~0.878–0.880).

The classification report shows strongest performance on classes 1, 5, 7, 8, and 9 (>0.96 F1). The weakest class remains 6 (Shirt), which is still commonly confused with 0 (T-shirt/Top), and class 4 (Coat) with 2 (Pullover). These confusions are expected given 28×28 grayscale images where silhouettes dominate over texture. Data augmentation (small rotations/translations) likely improved generalisation over the baselines.


In [13]:
results = pd.DataFrame(
    {
        "model": ["Deployed (pretrained)", "Local SavedModel", "Scratch CNN (yours)"],
        "accuracy": [
            float((y_pred_deployed == y_test).mean()),
            float(acc_local),
            float(test_acc),
        ],
    }
).sort_values("accuracy", ascending=False).reset_index(drop=True)
results

,model,accuracy
0,Scratch CNN (yours),0.9197
1,Local SavedModel,0.8803
2,Deployed (pretrained),0.8779


### 4.2 Summary

The scratch CNN is the top performer (\~0.922), ahead of the deployed (\~0.878) and local SavedModel (\~0.880). The gain is consistent with higher capacity (two conv blocks + dense 128) and the use of mild augmentation, which reduces overfitting to class silhouettes. Baselines are simpler MLP-like models without augmentation, which explains their lower accuracy.


In [14]:
EXPORT_DIR = "/home/jupyter/fashion-mnist-trained"
tf.saved_model.save(model_scratch, EXPORT_DIR)
!tar -C "/home/jupyter" -czf "/home/jupyter/fashion-mnist-trained.tar.gz" "fashion-mnist-trained"
print("SavedModel:", EXPORT_DIR)
print("Archive:   /home/jupyter/fashion-mnist-trained.tar.gz")

SavedModel: /home/jupyter/fashion-mnist-trained
Archive:   /home/jupyter/fashion-mnist-trained.tar.gz


In [15]:
storage.Client(project=PROJECT_ID).bucket(GCS_BUCKET)\
    .blob("fashion-mnist-trained.tar.gz")\
    .upload_from_filename("/home/jupyter/fashion-mnist-trained.tar.gz")
print(f"gs://{GCS_BUCKET}/fashion-mnist-trained.tar.gz")

gs://com3021-fmnist-rps210-bucket/fashion-mnist-trained.tar.gz
